In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import scene_generation.core as core_mod
import time
import json

from pathlib import Path
from scene_generation.core import Scene
from scene_generation.utils import rect_from_point_and_size
from collections import Counter
from matplotlib.patches import Patch
from PIL import Image, ImageDraw
from sionna.rt import scene, preview

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
relative_to_lidar_osm = []

lidar_osm_hag_perc = []
lidar_osm_height_perc = []
lidar_osm_building_levels_perc = []
lidar_osm_random_fallback_perc = []
overture_height_perc = []
overture_num_floors_perc = []
overture_random_fallback_perc = []

overall_mean_abs_diff = []
overall_max_abs_diff = []
lidar_outside_explicit_overture_height = []

lidar_osm_scene_gen_errors = []
overture_scene_gen_errors = []

STATS_FILE = Path("./stats_progress_with_parts_overture_lidar.json")

In [3]:
def save_stats():
    data = {
        "relative_to_lidar_osm": relative_to_lidar_osm,
        "lidar_osm_hag_perc": lidar_osm_hag_perc,
        "lidar_osm_height_perc": lidar_osm_height_perc,
        "lidar_osm_building_levels_perc": lidar_osm_building_levels_perc,
        "lidar_osm_random_fallback_perc": lidar_osm_random_fallback_perc,
        "overture_height_perc": overture_height_perc,
        "overture_num_floors_perc": overture_num_floors_perc,
        "overture_random_fallback_perc": overture_random_fallback_perc,
        "overall_mean_abs_diff": overall_mean_abs_diff,
        "overall_max_abs_diff": overall_max_abs_diff,
        "lidar_outside_explicit_overture_height": lidar_outside_explicit_overture_height,
        "lidar_osm_scene_gen_errors": lidar_osm_scene_gen_errors,
        "overture_scene_gen_errors": overture_scene_gen_errors
    }
    tmp = STATS_FILE.with_suffix(".tmp")
    STATS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(tmp, "w") as f:
        json.dump(data, f)
    os.replace(tmp, STATS_FILE)

In [4]:
def load_stats():
    if not STATS_FILE.exists():
        return
    with open(STATS_FILE, "r") as f:
        data = json.load(f)
    relative_to_lidar_osm.extend(data.get("relative_to_lidar_osm", []))
    lidar_osm_hag_perc.extend(data.get("lidar_osm_hag_perc", []))
    lidar_osm_height_perc.extend(data.get("lidar_osm_height_perc", []))
    lidar_osm_building_levels_perc.extend(data.get("lidar_osm_building_levels_perc", []))
    lidar_osm_random_fallback_perc.extend(data.get("lidar_osm_random_fallback_perc", []))
    overture_height_perc.extend(data.get("overture_height_perc", []))
    overture_num_floors_perc.extend(data.get("overture_num_floors_perc", []))
    overture_random_fallback_perc.extend(data.get("overture_random_fallback_perc", []))
    overall_mean_abs_diff.extend(data.get("overall_mean_abs_diff", []))
    overall_max_abs_diff.extend(data.get("overall_max_abs_diff", []))
    lidar_outside_explicit_overture_height.extend(data.get("lidar_outside_explicit_overture_height", []))
    lidar_osm_scene_gen_errors.extend(data.get("lidar_osm_scene_gen_errors", []))
    overture_scene_gen_errors.extend(data.get("overture_scene_gen_errors", []))


In [5]:
load_stats()

In [6]:
def run_analysis(CENTER_LON, CENTER_LAT, placename):    

    SCENE_WIDTH  = 500   # east–west extent, metres
    SCENE_HEIGHT = 500   # north–south extent, metres

    DATA_DIR_LIDAR_OSM = f"./scenes/{placename}_lidar_osm"
    DATA_DIR_OVERTURE = f"./scenes/{placename}_overture"

    OSM_SERVER = "http://10.237.198.210:3452/api/interpreter"
    # can use own private server or the public server, which will be slower after many requests: 
    # "https://overpass-api.de/api/interpreter"

    for out_dir in (DATA_DIR_LIDAR_OSM, DATA_DIR_OVERTURE):
        os.makedirs(out_dir, exist_ok=True)
        print(f"Output directory: {os.path.abspath(out_dir)}")

    def summarize_height_sources(mode, height_sources):
        """Summarize counts and percentages of buildings by selected height source."""
        total_buildings = sum(height_sources.values())
        rows = []

        for source, building_count in height_sources.most_common():
            building_percentage = (
                (building_count / total_buildings) * 100 if total_buildings else 0.0
            )
            rows.append(
                {
                    "mode": mode,
                    "height_source": source,
                    "building_count": building_count,
                    "building_percentage": f"{building_percentage:.2f}%",
                }
            )

        return pd.DataFrame(
            rows,
            columns=["mode", "height_source", "building_count", "building_percentage"],
        )

    def iter_polygons(geometry):
        """Yield polygon parts from a Shapely Polygon or MultiPolygon."""
        if geometry is None or geometry.is_empty:
            return

        if geometry.geom_type == "Polygon":
            yield geometry
        elif geometry.geom_type == "MultiPolygon":
            yield from geometry.geoms

    def rasterize_height_source_mask(records, height_source, shape, ground_bounds):
        """Rasterize footprints for one height source onto a building-map grid."""
        min_x, _, _, max_y = ground_bounds
        mask_image = Image.new("1", (shape[1], shape[0]), 0)
        draw = ImageDraw.Draw(mask_image)

        for record in records:
            if record["height_source"] != height_source:
                continue

            for polygon in iter_polygons(record["footprint"]):
                pixel_coords = [(x - min_x, max_y - y) for x, y in polygon.exterior.coords]
                draw.polygon(pixel_coords, outline=1, fill=1)

        return np.array(mask_image, dtype=bool)

    def generate_scene(mode, out_dir, *, track_height_sources=False):
        """Generate the scene and optionally count selected height sources."""
        scene_polygon = rect_from_point_and_size(
        CENTER_LON, CENTER_LAT, "center", SCENE_WIDTH, SCENE_HEIGHT
        )

        height_sources = []
        height_source_records = []
        original_resolve = core_mod.resolve_building_height

        def logging_resolve_building_height(*args, **kwargs):
            kwargs = dict(kwargs)
            kwargs["return_source"] = True
            height, metadata = original_resolve(*args, **kwargs)
            source = metadata.get("source", "unknown")
            footprint = args[1] if len(args) > 1 else kwargs.get("building_polygon")
            height_sources.append(source)
            height_source_records.append(
                {
                    "mode": mode,
                    "height_source": source,
                    "height_m": height,
                    "footprint": footprint,
                }
            )
            return height

        if track_height_sources:
            core_mod.resolve_building_height = logging_resolve_building_height

        try:
            scene = Scene()
            # print(scene_polygon)
            building_height_map = scene(
                points=scene_polygon,
                data_dir=out_dir,
                osm_server_addr=OSM_SERVER,
                hag_tiff_path=None,  # None lets lidar-osm create/reuse out_dir/test_hag.tif
                ground_material_type="mat-itu_wet_ground",
                rooftop_material_type="mat-itu_metal",
                wall_material_type="mat-itu_concrete",
                generate_building_map=True,
                building_height_mode=mode,
                # lidar_terrain=False,
                # dem_terrain=False,
            )
            ground_bounds = scene._ground_polygon_envelope_UTM.bounds
            if track_height_sources:
                core_mod.resolve_building_height = original_resolve

        except Exception as e:
            print(f"Error occurred while generating scene: {e}")
            if mode == "lidar-osm":
                lidar_osm_scene_gen_errors.append(placename)
            else:
                overture_scene_gen_errors.append(placename)
            save_stats()
            if track_height_sources:
                core_mod.resolve_building_height = original_resolve
            return None, None, None, None
        return building_height_map, Counter(height_sources), height_source_records, ground_bounds


    scene_generation_runtimes = []

    lidar_start_time = time.perf_counter()
    lidar_height_map, lidar_height_sources, lidar_height_source_records, lidar_ground_bounds = generate_scene(
        "lidar-osm", DATA_DIR_LIDAR_OSM, track_height_sources=True
    )
    lidar_end_time = time.perf_counter()

    overture_start_time = time.perf_counter()
    overture_height_map, overture_height_sources, overture_height_source_records, overture_ground_bounds = generate_scene(
        "overture", DATA_DIR_OVERTURE, track_height_sources=True
    )
    overture_end_time = time.perf_counter()

    if not lidar_ground_bounds:
        print("Failed to generate LiDAR-OSM scene.")
    if not overture_ground_bounds:
        print("Failed to generate Overture scene.")
    if not lidar_ground_bounds or not overture_ground_bounds:
        return

    scene_generation_runtimes.append(
        {"mode": "lidar-osm", "runtime_seconds": lidar_end_time - lidar_start_time}
    )
    scene_generation_runtimes.append(
        {"mode": "overture", "runtime_seconds": overture_end_time - overture_start_time}
    )

    print("Scene generation complete!")
    scene_generation_runtime_summary = pd.DataFrame(scene_generation_runtimes)
    lidar_runtime_seconds = scene_generation_runtime_summary.loc[
        scene_generation_runtime_summary["mode"] == "lidar-osm",
        "runtime_seconds",
    ].iloc[0]
    scene_generation_runtime_summary["runtime_seconds"] = scene_generation_runtime_summary[
        "runtime_seconds"
    ].round(3)
    scene_generation_runtime_summary["relative_to_lidar_osm"] = (
        scene_generation_runtime_summary["runtime_seconds"] / lidar_runtime_seconds
    ).round(2)

    relative_to_lidar_osm.append(scene_generation_runtime_summary["relative_to_lidar_osm"].iloc[1])
    save_stats()

    
    display(scene_generation_runtime_summary)

    height_source_summary = pd.concat(
        [
            summarize_height_sources("lidar-osm", lidar_height_sources),
            
            summarize_height_sources("overture", overture_height_sources),
        ],
        ignore_index=True,
    )
    display(height_source_summary)

    total_lidar = sum(lidar_height_sources.values())
    lidar_source_percents = {
        "hag": lidar_osm_hag_perc,
        "osm:height": lidar_osm_height_perc,
        "osm:building:levels": lidar_osm_building_levels_perc,
        "fallback:random": lidar_osm_random_fallback_perc,
    }
    for source, target_list in lidar_source_percents.items():
        perc = (
            100.0 * lidar_height_sources.get(source, 0) / total_lidar
            if total_lidar
            else 0.0
        )
        target_list.append(perc)

    save_stats()

    total_overture = sum(overture_height_sources.values())
    overture_source_percents = {
        "overture:height": overture_height_perc,
        "overture:num_floors": overture_num_floors_perc,
        "fallback:random": overture_random_fallback_perc,
    }
    for source, target_list in overture_source_percents.items():
        perc = (
            100.0 * overture_height_sources.get(source, 0) / total_overture
            if total_overture
            else 0.0
        )
        target_list.append(perc)

    save_stats()

    # Compare final building-height maps from the two modes.
    lidar_map = np.load(Path(DATA_DIR_LIDAR_OSM) / "2D_Building_Height_Map.npy")
    overture_map = np.load(Path(DATA_DIR_OVERTURE) / "2D_Building_Height_Map.npy")

    diff = lidar_map.astype(float) - overture_map.astype(float)
    building_mask = (lidar_map > 0) | (overture_map > 0)
    mean_abs_diff = np.mean(np.abs(diff[building_mask])) if building_mask.any() else 0.0
    max_abs_diff = np.max(np.abs(diff)) if diff.size else 0.0
    lidar_hag_mask = rasterize_height_source_mask(
        lidar_height_source_records,
        "hag",
        lidar_map.shape,
        lidar_ground_bounds,
    )
    overture_explicit_height_mask = rasterize_height_source_mask(
        overture_height_source_records,
        "overture:height",
        lidar_map.shape,
        overture_ground_bounds,
    )
    building_pixel_count = int(np.count_nonzero(building_mask))
    lidar_hag_not_overture_explicit_height_pixels = int(
            np.count_nonzero(lidar_hag_mask & ~overture_explicit_height_mask)
    )
    lidar_hag_not_overture_explicit_height_percent = (
            100.0
            * lidar_hag_not_overture_explicit_height_pixels
            / building_pixel_count
            if building_pixel_count
            else 0.0
    )
    overall_mean_abs_diff.append(float(mean_abs_diff))
    overall_max_abs_diff.append(float(max_abs_diff))
    lidar_outside_explicit_overture_height.append(lidar_hag_not_overture_explicit_height_percent)
    save_stats()

    comparison = pd.DataFrame(
        [
            ("same raster", np.array_equal(lidar_map, overture_map)),
            ("mean abs diff on building pixels (m)", round(float(mean_abs_diff), 3)),
            ("max abs diff (m)", round(float(max_abs_diff), 3)),
            (
                "LiDAR HAG pixels outside Overture explicit height (%)",
                round(float(lidar_hag_not_overture_explicit_height_percent), 3),
            ),
        ],
        columns=["check", "value"],
    )
    display(comparison)

    height_vmax = max(float(lidar_map.max()), float(overture_map.max()), 1.0)
    diff_abs_max = max(float(np.max(np.abs(diff))), 1.0) if diff.size else 1.0

    plt.show()

In [ ]:
# Scene center (only need to update CENTER_LON, CENTER_LAT, DATA_DIR_LIDAR_OSM, DATA_DIR_OVERTURE and run the rest of the cells to receive full analysis)
# DuPont Circle: -77.043446, 38.909647
# Duke Wilkinson area: -78.940297, 36.002556
# Flatiron Building: -73.9897, 40.7411

'''
21 areas used for initial analysis
areas_of_interest = [
    [-77.043446, 38.909647, "dupont"],
    [-78.940297, 36.002556, "wilkinson"],
    [-73.9897, 40.7411, "flatiron"],
    [-118.40036, 34.07362, "beverlyhills"],
    [-87.5939377, 41.7942008, "hydepark"],
    [-80.237709, 25.777643, "littlehavanamiami"],
    [-75.1896236, 39.9492795, "universitycityphilly"],
    [-95.388992, 29.760427, "houston"],
    [-96.7900708, 32.7849914, "deepellumdallas"],
    [-77.024698, 38.879393, "dcwharf"],
    [-83.3623853, 33.5684599, "buckheadatlanta"],
    [-112.074036, 33.448376, "phoenix"],
    [-82.99611, 42.36028, "indianvillagedetroit"],
    [-122.316456, 47.622942, "capitolhillseattle"],
    [-122.41448, 37.79323, "nobhillsanfrancisco"],
    [-117.142586, 32.730831, "balboaparksandiego"],
    [-93.258133, 44.986656, "minneapolis"],
    [-82.4573, 27.9942, "seminoleheightstampa"],
    [-104.991531, 39.742043, "denver"],
    [-117.396156, 33.953350, "riverside"],
    [-76.6317, 39.3317, "hampdenbaltimore"]
]
'''

# use cities.json (obtained from https://gist.github.com/Miserlou/c5cd8364bf9b2420bb29)
# to parse for U.S. urban areas
with open("cities.json", "r") as f:
    cities = json.load(f)

areas_of_interest = []
for i in range(100):
   areas_of_interest.append([cities[i]["longitude"], cities[i]["latitude"], cities[i]["city"].replace(" ", "_")])
for i in range(len(cities) - 1, len(cities) - 101, -1):
    areas_of_interest.append([cities[i]["longitude"], cities[i]["latitude"], cities[i]["city"].replace(" ", "_")])

for CENTER_LON, CENTER_LAT, placename in areas_of_interest:
    run_analysis(CENTER_LON, CENTER_LAT, placename)

Output directory: /home/rt279/geo2sigmap-fork/research/examples/scenes/New_York_lidar_osm
Output directory: /home/rt279/geo2sigmap-fork/research/examples/scenes/New_York_overture
Loading local 3DEP dataset polygons...
Done. 3DEP polygons downloaded and projected to  EPSG:32618
Area of Interest: POLYGON ((-8238803.425536943 4969578.742170027, -8238792.273554624 4970570.867185732, -8237803.943548938 4970559.637150297, -8237815.162102232 4969567.515047306, -8238803.425536943 4969578.742170027))
NY_NewYorkCity
https://s3-us-west-2.amazonaws.com/usgs-lidar-public/NY_NewYorkCity/ept.json
Found 1 intersecting datasets
Successfully generated HAG data


Parsing buildings: 100%|██████████| 156/156 [00:01<00:00, 102.48it/s]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parsing buildings:  28%|██▊       | 83/300 [00:00<00:00, 411.63it/s]Building part f76ae995-f55a-321f-bc0b-700e563d160e top height 44.00 exceeds parent building d058153b-c5a7-40f7-a4b6-36005c9bd49c top height 40.00; treating min_height/min_floor as 0 for the part
Building part e89a56ec-3ddb-326b-8fea-50f9608350a1 top height 74.50 exceeds parent building f8437dfe-0b14-4e58-96bb-bba278fe1163 top height 22.63; treating min_height/min_floor as 0 for the part
Parsing buildings:  42%|████▏     | 125/300 [00:00<00:00, 411.28it/s]Building part 15b51636-fe89-3345-9336-1e84a5664971 top height 98.50 exceeds parent building c1298716-7c0f-420d-bc21-36eb92ebe185 top height 35.00; treating min_height/min_floor as 0 for the part
Building part d0bf1fb1-7844-3faa-adba-e349d3e7f027 top height 55.00 exceeds parent building f40050c5-ff92-401f-8570-633169e52a92 top height 22.38; treating min_height/min_floor as 0 for the part
Building part 16d6ddf4-e4ec-3ffa-b9f4-94a915886b36 top height 37.00 exceeds parent 

...

,mode,height_source,building_count,building_percentage
0,lidar-osm,hag,13,65.00%
1,lidar-osm,fallback:random,7,35.00%
2,overture,overture:height,27,84.38%
3,overture,fallback:random,5,15.62%


,check,value
0,same raster,False
1,mean abs diff on building pixels (m),3.462
2,max abs diff (m),19.0
3,LiDAR HAG pixels outside Overture explicit hei...,10.8


Output directory: /home/rt279/geo2sigmap-fork/research/examples/scenes/Montclair_lidar_osm
Output directory: /home/rt279/geo2sigmap-fork/research/examples/scenes/Montclair_overture
Loading local 3DEP dataset polygons...
Done. 3DEP polygons downloaded and projected to  EPSG:32611
Area of Interest: POLYGON ((-13101615.46741035 4038756.962461769, -13101621.595293948 4039665.979982502, -13100716.736650785 4039672.1128787715, -13100710.656709839 4038763.093834516, -13101615.46741035 4038756.962461769))
USGS_LPC_CA_SanBernardinoCo_AreaA_2013_LAS_2018
https://s3-us-west-2.amazonaws.com/usgs-lidar-public/USGS_LPC_CA_SanBernardinoCo_AreaA_2013_LAS_2018/ept.json
USGS_LPC_CA_SoCal_Wildfires_B1_2018_LAS_2019
https://s3-us-west-2.amazonaws.com/usgs-lidar-public/USGS_LPC_CA_SoCal_Wildfires_B1_2018_LAS_2019/ept.json
Found 2 intersecting datasets
Successfully generated HAG data


Parsing buildings: 100%|██████████| 20/20 [00:00<00:00, 86.68it/s]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))